 ### Importing Libraries and Initial Setup


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from data_aggregation.creating_df_full import build_full_dataset
import json


---
### Data Preparation – Building the Full Dataset

In [ ]:
X, y, df_full = build_full_dataset(last_season=2025, n_seasons=16, return_full=True)


---
### Exporting the Dataset to CSV

In [ ]:
df_full.to_csv("databases/nba_dataset_2010_2025.csv", index=False)



---
### Quick Exploratory Data Analysis

##### Target Distribution

In [ ]:
import matplotlib.pyplot as plt
df_full[df_full["season"] < 2025]["target"].value_counts().sort_index().plot(kind="bar")
plt.title("Rozkład klas targetu (2010–2025)")
plt.xlabel("Target")
plt.ylabel("Liczba zawodników")
plt.show()


##### Feature Correlation Analysis

In [ ]:
correlations = df_full.corr(numeric_only=True)["target"].sort_values(ascending=False)
print(correlations.head(20))  # najważniejsze cechy


--- 
### Cross Validation - Comparison of Different Models

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from datetime import datetime
import mlflow
from mlflow.models import infer_signature

log_to_mlflow = False  # Zmień na False jeśli nie chcesz logować do MLflow

if log_to_mlflow:
    mlflow.set_tracking_uri("http://127.0.0.1:8080")
    mlflow.set_experiment("First Models Testing")


# Modele, które wymagają skalowania
needs_scaling = {"LinearSVM", "RBFSVM", "KNN", "MLP", "GaussianProcess", "LogisticRegression"}

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF

models = {
    "RandomForest": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=3000, class_weight="balanced", random_state=42),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=300, random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(alpha=1, max_iter=1000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=3),
    "LinearSVM": SVC(kernel="linear", C=0.025, probability=True, random_state=42),
    "RBFSVM": SVC(gamma=2, C=1, probability=True, random_state=42),
    "GaussianProcess": GaussianProcessClassifier(1.0 * RBF(1.0), random_state=42),
    "DecisionTree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "NaiveBayes": GaussianNB(),
    "QDA": QuadraticDiscriminantAnalysis()
}

#Opcjonalnie XGBoost
try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss", random_state=42)
except ImportError:
    print("⚠️ XGBoost not installed — skipping")


# Załaduj dane z sezonów 2010–2025
df_full = pd.read_csv("databases/nba_dataset_2010_2025.csv")
X = df_full.drop(columns=["Player", "Team", "Pos", "season", "target"])
y = df_full["target"]

# Wyniki
all_results = []

# Walidacja sezon-po-sezonie dla każdego modelu
for model_name, model in models.items():
    print(f"\n🔍 Testujemy model: {model_name}")
    
    if log_to_mlflow:
        if mlflow.active_run():
            mlflow.end_run()
        mlflow_run = mlflow.start_run(run_name=model_name)
        mlflow.log_param("model_name", model_name)
        
    for season in range(2015, 2025):
    
        mask_train = df_full["season"] == season
        mask_test = df_full["season"] == season + 1

        X_train = X[mask_train]
        y_train = y[mask_train]
        X_test = X[mask_test]
        y_test = y[mask_test]

        # Skalowanie tylko tam, gdzie potrzebne
        if model_name in needs_scaling:
            pipeline = make_pipeline(StandardScaler(), model)
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
        else:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

        report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

        all_results.append({
            "model": model_name,
            "train_season": season,
            "test_season": season + 1,
            "accuracy": report["accuracy"],
            "macro_f1": report["macro avg"]["f1-score"],
            "weighted_f1": report["weighted avg"]["f1-score"]
        })


        if log_to_mlflow:
            mlflow.log_metric("accuracy", report["accuracy"], step=season)
            mlflow.log_metric("macro_f1", report["macro avg"]["f1-score"], step=season)
            mlflow.log_metric("weighted_f1", report["weighted avg"]["f1-score"], step=season)


# Wyniki jako DataFrame
df_results = pd.DataFrame(all_results)
df_results = df_results.round(3)

# Zapisz wyniki do CSV
df_results.to_csv("testing_models_results/first_models_comparison_results.csv", index=False)

print("Wyniki zapisane do pliku: first_models_comparison_results.csv")
print("\nPodsumowanie:")
#print(df_results)

# if log_to_mlflow:
#     mlflow.log_artifact("testing_models_results/first_models_comparison_results.csv")
#     mlflow.end_run()



In [ ]:
import mlflow  
mlflow.end_run()


---
### Models Training and Evaluation

#### XGBoost Hyperparameter Tuning with GridSearchCV

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

xgb = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss", random_state=42)

# 1. Załaduj dane z sezonów 2010–2025
df_full = pd.read_csv("nba_dataset_2010_2025.csv")
X = df_full.drop(columns=["Player", "Team", "Pos", "season", "target"])
y = df_full["target"]


# Podział: 2010–2024 → trening, 2025 → predykcja
mask_train = df_full["season"] < 2025
mask_pred = df_full["season"] == 2025

X_train = X[mask_train]
y_train = y[mask_train]

param_grid = {
    "n_estimators": [100, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.9, 1.0],
    "gamma": [0, 1, 5],  # kontrola złożoności drzewa
    "reg_alpha": [0, 0.1, 1],  # L1 regularization
    "reg_lambda": [1, 3, 5]    # L2 regularization
}

grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    verbose=2,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Najlepsze parametry:", grid.best_params_)
best_model = grid.best_estimator_


#### XGBoost Hyperparameter Tuning with RandomizedSearchCV

In [ ]:
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
import mlflow
import mlflow.sklearn  # do logowania modelu

# === Konfiguracja MLflow ===
log_to_mlflow = False

if log_to_mlflow:
    mlflow.set_tracking_uri("http://127.0.0.1:8080")
    mlflow.set_experiment("XGBoost Hyperparam Tuning")

# === Model ===
xgb = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss", random_state=42)

df_full = pd.read_csv("databases/nba_dataset_2010_2025.csv")

# === Podział ===
df_majority = df_full[(df_full["season"] < 2025) & (df_full["target"] == 0)]
df_minority = df_full[(df_full["season"] < 2025) & (df_full["target"] > 0)]

# === Undersampling klasy 0 ===
df_majority_sampled = df_majority.sample(frac=0.5, random_state=42)
df_train_balanced = pd.concat([df_majority_sampled, df_minority])

drop_cols = ["Player", "Team", "Pos", "season", "target"]
X_train = df_train_balanced.drop(columns=drop_cols)
y_train = df_train_balanced["target"]

# === Parametry ===
param_grid = {
    "n_estimators": [100, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.9, 1.0],
    "gamma": [0, 1, 5],
    "reg_alpha": [0, 0.1, 1],
    "reg_lambda": [1, 3, 5]
}

# === Random Search ===
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=50,
    scoring="f1_macro",
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

# === Trenowanie z MLflow ===
if log_to_mlflow:
    with mlflow.start_run(run_name="XGB_RandomSearch"):
        random_search.fit(X_train, y_train)

        # log parametry i wynik
        mlflow.log_params(random_search.best_params_)
        mlflow.log_metric("best_f1_macro", random_search.best_score_)

        # zapisz model
        mlflow.sklearn.log_model(random_search.best_estimator_, artifact_path="model")

        print("Best parameters:", random_search.best_params_)

else:
    random_search.fit(X_train, y_train)
    print("Best parameters:", random_search.best_params_)


#### XGBoost Training and Generating Prediction

In [ ]:
import pandas as pd
import json
from xgboost import XGBClassifier
import mlflow
import mlflow.sklearn

# === MLflow config ===
log_to_mlflow = False

if log_to_mlflow:
    mlflow.set_tracking_uri("http://127.0.0.1:8080")
    mlflow.set_experiment("NBA Final Prediction 2025")

# === 1. Załaduj dane ===
df_full = pd.read_csv("databases/nba_dataset_2010_2025.csv")
X = df_full.drop(columns=["Player", "Team", "Pos", "season", "target"])
y = df_full["target"]

mask_train = df_full["season"] < 2025
mask_pred = df_full["season"] == 2025

X_train = X[mask_train]
y_train = y[mask_train]
X_pred = X[mask_pred]
players_pred = df_full[mask_pred]["Player"].reset_index(drop=True)
is_rookie = df_full[mask_pred]["is_rookie"].reset_index(drop=True)

# === 2. Model ===
model = XGBClassifier(
    subsample=1.0,
    colsample_bytree=0.9,
    reg_lambda=5,
    reg_alpha=0,
    n_estimators=100,
    max_depth=5,
    learning_rate=0.01,
    gamma=1,
    eval_metric="mlogloss",
    random_state=42
)

# === 3. Trenowanie + MLflow logging ===
if log_to_mlflow:
    with mlflow.start_run(run_name="XGB_Prediction_2025"):
        mlflow.log_params(model.get_params())
        mlflow.set_tag("model_type", "XGBClassifier")
        mlflow.set_tag("prediction_season", "2025")

        model.fit(X_train, y_train)
        mlflow.sklearn.log_model(model, artifact_path="model")

        # === 4. Predykcja ===
        probas = model.predict_proba(X_pred)
        df_pred = pd.DataFrame(probas, columns=model.classes_)
        df_pred["Player"] = players_pred
        df_pred["is_rookie"] = is_rookie

        results = {
            "first all-nba team": df_pred.sort_values(1, ascending=False)["Player"].head(5).tolist(),
            "second all-nba team": df_pred.sort_values(2, ascending=False)["Player"].head(5).tolist(),
            "third all-nba team": df_pred.sort_values(3, ascending=False)["Player"].head(5).tolist(),
            "first rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(4, ascending=False)["Player"].head(5).tolist(),
            "second rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(5, ascending=False)["Player"].head(5).tolist()
        }

        output_path = "classification_result_2025.json"
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

        mlflow.log_artifact(output_path)

else:
    model.fit(X_train, y_train)
    probas = model.predict_proba(X_pred)
    df_pred = pd.DataFrame(probas, columns=model.classes_)
    df_pred["Player"] = players_pred
    df_pred["is_rookie"] = is_rookie

    results = {
        "first all-nba team": df_pred.sort_values(1, ascending=False)["Player"].head(5).tolist(),
        "second all-nba team": df_pred.sort_values(2, ascending=False)["Player"].head(5).tolist(),
        "third all-nba team": df_pred.sort_values(3, ascending=False)["Player"].head(5).tolist(),
        "first rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(4, ascending=False)["Player"].head(5).tolist(),
        "second rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(5, ascending=False)["Player"].head(5).tolist()
    }

    with open("classification_result_2025.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

print("Gotowe")


Uzyskany score:

- Total score: 256/450
- first all-nba team: 68 points
- second all-nba team: 56 points
- third all-nba team: 8 points
- first rookie all-nba team: 68 points
- second rookie all-nba team: 56 points

#### Class Balancing – Undersampling Dataset, XGBoost Training and Prediction

Model uczony na niezbalansowanych danych miałby tendencję do faworyzowania klasy większościowej (brak nagrody), co skutkowałoby niską czułością względem klas pozytywnych (czyli faktycznych wyróżnień). Aby temu zapobiec:

1. **Undersampling klasy 0** – losowo wybierane jest 50% przypadków z klasy `target == 0` (zawodnicy bez nagród).
2. **Zachowanie wszystkich przypadków klasy mniejszościowej** – `target > 0` (czyli zawodnicy nagrodzeni).
3. **Łączenie** tych zbiorów daje bardziej zrównoważony zbiór treningowy.

Efekt:
- Zwiększenie reprezentacji przykładów pozytywnych w zbiorze uczącym.
- Lepsza zdolność modelu do rozróżniania i typowania zawodników, którzy **mogą** trafić do All-NBA lub All-Rookie Teams.
- Zmniejszenie ryzyka biasu modelu w stronę klasy "brak nagrody".

In [ ]:
import pandas as pd
import json
from xgboost import XGBClassifier
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

# === MLflow setup ===
log_to_mlflow = False

if log_to_mlflow:
    mlflow.set_tracking_uri("http://127.0.0.1:8080")
    mlflow.set_experiment("NBA Final Prediction 2025")

# === Dane ===
df_full = pd.read_csv("databases/nba_dataset_2010_2025.csv")

# === Podział ===
df_majority = df_full[(df_full["season"] < 2025) & (df_full["target"] == 0)]
df_minority = df_full[(df_full["season"] < 2025) & (df_full["target"] > 0)]
df_majority_sampled = df_majority.sample(frac=0.5, random_state=42)
df_train_balanced = pd.concat([df_majority_sampled, df_minority])

drop_cols = ["Player", "Team", "Pos", "season", "target"]
X_train = df_train_balanced.drop(columns=drop_cols)
y_train = df_train_balanced["target"]

# === Dane do predykcji ===
X_pred = df_full[df_full["season"] == 2025].drop(columns=drop_cols)
players_pred = df_full[df_full["season"] == 2025]["Player"].reset_index(drop=True)
is_rookie = df_full[df_full["season"] == 2025]["is_rookie"].reset_index(drop=True)

# === Model ===
model = XGBClassifier(
    n_estimators=700,
    colsample_bytree=0.9,
    max_bin=128,
    subsample=0.7,
    max_depth=5,
    learning_rate=0.01,
    grow_policy="depthwise",
    reg_alpha=0.01,
    reg_lambda=10,
    min_child_weight=1,
    gamma=0.5,
    tree_method="hist",
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42
)


# === Trening i predykcja z logowaniem ===
if log_to_mlflow:
    with mlflow.start_run(run_name="V2_XGB_Undersampling_Prediction_2025"):
        mlflow.log_params(model.get_params())
        mlflow.set_tag("model_type", "XGBClassifier")
        mlflow.set_tag("prediction_season", "2025")

        model.fit(X_train, y_train)
        probas = model.predict_proba(X_pred)

        # Infer signature
        signature = infer_signature(X_train, model.predict(X_train))
        input_example = X_train.iloc[:1]

        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            signature=signature,
            input_example=input_example
        )

        df_pred = pd.DataFrame(probas, columns=model.classes_)
        df_pred["Player"] = players_pred
        df_pred["is_rookie"] = is_rookie

        results = {
            "first all-nba team": df_pred.sort_values(1, ascending=False)["Player"].head(5).tolist(),
            "second all-nba team": df_pred.sort_values(2, ascending=False)["Player"].head(5).tolist(),
            "third all-nba team": df_pred.sort_values(3, ascending=False)["Player"].head(5).tolist(),
            "first rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(4, ascending=False)["Player"].head(5).tolist(),
            "second rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(5, ascending=False)["Player"].head(5).tolist()
        }

        output_path = "classification_result_2025.json"
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

        mlflow.log_artifact(output_path)

else:
    model.fit(X_train, y_train)
    probas = model.predict_proba(X_pred)

    df_pred = pd.DataFrame(probas, columns=model.classes_)
    df_pred["Player"] = players_pred
    df_pred["is_rookie"] = is_rookie

    results = {
        "first all-nba team": df_pred.sort_values(1, ascending=False)["Player"].head(5).tolist(),
        "second all-nba team": df_pred.sort_values(2, ascending=False)["Player"].head(5).tolist(),
        "third all-nba team": df_pred.sort_values(3, ascending=False)["Player"].head(5).tolist(),
        "first rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(4, ascending=False)["Player"].head(5).tolist(),
        "second rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(5, ascending=False)["Player"].head(5).tolist()
    }

    with open("classification_result_2025.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

print("Gotowe")


Uzyskany score:
- Total score: 250/450
- first all-nba team: 68 points
- second all-nba team: 48 points
- third all-nba team: 22 points
- first rookie all-nba team: 56 points
- second rookie all-nba team: 56 points

##### XGboost Undersampling Hyperparam Tuning 

In [ ]:
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
import mlflow
import mlflow.sklearn  # do logowania modelu

# === Konfiguracja MLflow ===
log_to_mlflow = False

if log_to_mlflow:
    mlflow.set_tracking_uri("http://127.0.0.1:8080")
    mlflow.set_experiment("XGBoost Hyperparam Tuning")

# === Model ===
xgb = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss", random_state=42)

# === Dane ===
df_full = pd.read_csv("databases/nba_dataset_2010_2025.csv")

# === Podział ===
df_majority = df_full[(df_full["season"] < 2025) & (df_full["target"] == 0)]
df_minority = df_full[(df_full["season"] < 2025) & (df_full["target"] > 0)]
df_majority_sampled = df_majority.sample(frac=0.5, random_state=42)
df_train_balanced = pd.concat([df_majority_sampled, df_minority])

drop_cols = ["Player", "Team", "Pos", "season", "target"]
X_train = df_train_balanced.drop(columns=drop_cols)
y_train = df_train_balanced["target"]

# === Dane do predykcji ===
X_pred = df_full[df_full["season"] == 2025].drop(columns=drop_cols)
players_pred = df_full[df_full["season"] == 2025]["Player"].reset_index(drop=True)
is_rookie = df_full[df_full["season"] == 2025]["is_rookie"].reset_index(drop=True)

# === Parametry ===
param_grid = {
    "n_estimators": [100, 200, 300, 400, 500, 700],
    "learning_rate": [0.005, 0.01, 0.03, 0.05, 0.1],
    "max_depth": [3, 4, 5, 6, 7, 9],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "gamma": [0, 0.5, 1, 2, 5, 10],
    "reg_alpha": [0, 0.01, 0.1, 1, 10],
    "reg_lambda": [0.1, 1, 3, 5, 10],
    "min_child_weight": [1, 3, 5, 7, 10],
    "max_bin": [128, 256, 512],
    "grow_policy": ["depthwise", "lossguide"],
    "tree_method": ["hist", "exact", "auto"]
}

# === Random Search ===
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=50,
    scoring="f1_macro",
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

# === Trenowanie z MLflow ===
if log_to_mlflow:
    with mlflow.start_run(run_name="XGB_UnderSampled_RandomSearch_2025"):
        random_search.fit(X_train, y_train)

        # log parametry i wynik
        mlflow.log_params(random_search.best_params_)
        mlflow.log_metric("best_f1_macro", random_search.best_score_)

        # zapisz model
        mlflow.sklearn.log_model(random_search.best_estimator_, artifact_path="model")

        print("Best parameters:", random_search.best_params_)

else:
    random_search.fit(X_train, y_train)
    print("Best parameters:", random_search.best_params_)


 #### Cascade Model, XGBoost – Hyperparameter Tuning with RandomizedSearchCV - Double Cross-Validation 

 Stage 1: Binary Classification (Award: Yes/No)
 
  Stage 2: Multiclass Classification (Team Selection)

In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import RandomizedSearchCV
import mlflow

# === MLflow config ===
mlflow.set_tracking_uri("http://127.0.0.1:8080")
mlflow.set_experiment("XGBoost Hyperparam Tuning")

with mlflow.start_run(run_name="XGBoost_2Stages_RandomSearch_2010-2024"):
    # === Dane ===
    df = pd.read_csv("databases/nba_dataset_2010_2025.csv")
    drop_cols = ["Player", "Team", "Pos", "season", "target"]
    df["target_binary"] = (df["target"] > 0).astype(int)

    train_mask = df["season"] < 2025
    X_train = df[train_mask].drop(columns=drop_cols)
    y_train_bin = df[train_mask]["target_binary"]
    y_train_full = df[train_mask]["target"]

    # === Parametry do przeszukania ===
    param_distributions = {
        "n_estimators": [100, 200, 300, 500, 700],
        "learning_rate": [0.005, 0.01, 0.03, 0.05, 0.1],
        "max_depth": [3, 4, 5, 6, 7],
        "subsample": [0.6, 0.8, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0],
        "reg_alpha": [0, 0.01, 0.1, 1],
        "reg_lambda": [0.1, 0.5, 1, 3],
        "gamma": [0, 0.5, 1, 5]
    }

    # === Etap 1 - klasyfikacja binarna ===
    model_bin = XGBClassifier(
        objective="binary:logistic",
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=42
    )

    search_stage1 = RandomizedSearchCV(
        estimator=model_bin,
        param_distributions=param_distributions,
        n_iter=50,
        scoring="recall",
        cv=3,
        verbose=2,
        n_jobs=-1
    )

    sample_weight_stage1 = compute_sample_weight("balanced", y_train_bin)
    search_stage1.fit(X_train, y_train_bin, sample_weight=sample_weight_stage1)

    mlflow.log_params({f"stage1__{k}": v for k, v in search_stage1.best_params_.items()})
    mlflow.log_metric("stage1_best_recall", search_stage1.best_score_)

    # === Etap 2 - klasyfikacja wieloklasowa ===
    class_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
    y_train_stage2 = y_train_full[y_train_bin == 1].map(class_map)
    X_train_stage2 = X_train[y_train_bin == 1]

    model_multi = XGBClassifier(
        objective="multi:softprob",
        num_class=5,
        use_label_encoder=False,
        eval_metric="mlogloss",
        random_state=42
    )

    search_stage2 = RandomizedSearchCV(
        estimator=model_multi,
        param_distributions=param_distributions,
        n_iter=50,
        scoring="f1_macro",
        cv=3,
        verbose=2,
        n_jobs=-1
    )

    search_stage2.fit(X_train_stage2, y_train_stage2)

    mlflow.log_params({f"stage2__{k}": v for k, v in search_stage2.best_params_.items()})
    mlflow.log_metric("stage2_best_f1_macro", search_stage2.best_score_)

    mlflow.set_tag("mode", "param_search_only")
    mlflow.set_tag("prediction_season", "2025")

print("Gotowe: najlepsze parametry i metryki zapisane do MLflow.")


##### Feature Importance Analysis – XGBoost

Stage 1: Binary Classification (Award: Yes/No)

In [ ]:
import xgboost as xgb
from xgboost import plot_importance
import matplotlib.pyplot as plt

# Jeśli masz model XGBoost, np. xgb_model = xgb.XGBClassifier().fit(...)
plot_importance(best_model_stage1, max_num_features=20, importance_type='gain')  # lub 'weight', 'cover'
plt.title("XGBoost Feature Importance")
plt.show()

Stage 2: Multiclass Classification (Team Selection)

In [ ]:
import xgboost as xgb
from xgboost import plot_importance
import matplotlib.pyplot as plt

# Jeśli masz model XGBoost, np. xgb_model = xgb.XGBClassifier().fit(...)
plot_importance(best_model_stage2, max_num_features=20, importance_type='gain')  # lub 'weight', 'cover'
plt.title("XGBoost Feature Importance")
plt.show()

 #### Cascade Model, XGBoost – Traning with assigned Hyperparameters and Prediction 

In [ ]:
import pandas as pd
import json
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
import mlflow
import mlflow.sklearn

# === MLflow setup ===
mlflow.set_tracking_uri("http://127.0.0.1:8080")
mlflow.set_experiment("NBA Final Prediction 2025")

with mlflow.start_run(run_name="V2_XGB_Cascade_2025"):
    # === 1. Wczytanie danych ===
    df = pd.read_csv("databases/nba_dataset_2010_2025.csv")
    drop_cols = ["Player", "Team", "Pos", "season", "target"]

    df["target_binary"] = (df["target"] > 0).astype(int)

    train_mask = df["season"] < 2025
    test_mask = df["season"] == 2025

    X_train = df[train_mask].drop(columns=drop_cols)
    X_test = df[test_mask].drop(columns=drop_cols)

    y_train_bin = df[train_mask]["target_binary"]
    y_train_full = df[train_mask]["target"]

    players_test = df[test_mask]["Player"].reset_index(drop=True)
    is_rookie = df[test_mask]["is_rookie"].reset_index(drop=True)

    # === 2. Etap 1: klasyfikacja binarna ===
    model_bin = XGBClassifier(
        reg_alpha=1,
        learning_rate=0.005,
        max_depth=5,
        subsample=0.6,
        colsample_bytree=1.0,
        n_estimators=300,
        gamma=5,
        reg_lambda=3,
        objective="binary:logistic",
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=42
    )

    mlflow.log_params({f"stage1__{k}": v for k, v in model_bin.get_params().items()})

    sample_weight_stage1 = compute_sample_weight("balanced", y_train_bin)
    model_bin.fit(X_train, y_train_bin, sample_weight=sample_weight_stage1)
    stage1_preds = model_bin.predict(X_test)

    # === 3. Etap 2: klasyfikacja wieloklasowa ===
    class_map = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
    inverse_class_map = {v: k for k, v in class_map.items()}

    X_train_stage2 = X_train[y_train_bin == 1]
    y_train_stage2 = y_train_full[y_train_bin == 1].map(class_map)

    X_test_stage2 = X_test[stage1_preds == 1].reset_index(drop=True)
    stage1_mask = pd.Series(stage1_preds == 1, index=players_test.index)

    players_stage2 = players_test[stage1_mask].reset_index(drop=True)
    is_rookie_stage2 = is_rookie[stage1_mask].reset_index(drop=True)

    model_multi = XGBClassifier(
        learning_rate=0.005,
        reg_alpha=0,
        n_estimators=100,
        reg_lambda=0.5,
        max_depth=3,
        colsample_bytree=0.6,
        subsample=0.6,
        gamma=1,
        objective="multi:softprob",
        num_class=5,
        use_label_encoder=False,
        eval_metric="mlogloss",
        random_state=42
    )

    mlflow.log_params({f"stage2__{k}": v for k, v in model_multi.get_params().items()})

    model_multi.fit(X_train_stage2, y_train_stage2)
    probas_stage2 = model_multi.predict_proba(X_test_stage2)

    # === 4. Budowanie piątek
    df_pred = pd.DataFrame(probas_stage2, columns=[inverse_class_map[i] for i in range(5)])
    df_pred["Player"] = players_stage2
    df_pred["is_rookie"] = is_rookie_stage2

    results = {}
    ordinal = ["first", "second", "third"]
    already_selected = set()

    for idx, class_id in enumerate([1, 2, 3]):
        available = df_pred[~df_pred["Player"].isin(already_selected)]
        top5 = available.sort_values(class_id, ascending=False).head(5)
        team_name = f"{ordinal[idx]} all-nba team"
        team_players = top5["Player"].tolist()
        results[team_name] = team_players
        already_selected.update(team_players)

    already_rookies = set()
    for idx, class_id in enumerate([4, 5]):
        available_rookies = df_pred[(df_pred["is_rookie"] == 1) & (~df_pred["Player"].isin(already_rookies))]
        top5 = available_rookies.sort_values(class_id, ascending=False).head(5)
        team_name = f"{ordinal[idx]} rookie all-nba team"
        team_players = top5["Player"].tolist()
        results[team_name] = team_players

    # === 5. Zapis JSON i log
    output_file = "classification_result_2025.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    mlflow.log_artifact(output_file)
    mlflow.set_tag("pipeline", "two_stage_prediction")
    mlflow.set_tag("prediction_season", "2025")

print("Gotowe.")


Uzyskany score:
- Total score: 316/450
- first all-nba team: 68 points
- second all-nba team: 56 points
- third all-nba team: 68 points
- first rookie all-nba team: 68 points
- second rookie all-nba team: 56 points

#### GradientBoosting Undersampling Hyperparameter Tuning with RandomizedSearchCV

In [ ]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
import mlflow
import mlflow.sklearn

# === MLflow config ===
mlflow.set_tracking_uri("http://127.0.0.1:8080")
mlflow.set_experiment("GBC Hyperparam Tuning")

# === Dane ===
df_full = pd.read_csv("databases/nba_dataset_2010_2025.csv")

df_majority = df_full[(df_full["season"] < 2025) & (df_full["target"] == 0)]
df_minority = df_full[(df_full["season"] < 2025) & (df_full["target"] > 0)]

df_majority_sampled = df_majority.sample(frac=0.5, random_state=42)
df_train_balanced = pd.concat([df_majority_sampled, df_minority])

drop_cols = ["Player", "Team", "Pos", "season", "target"]
X_train = df_train_balanced.drop(columns=drop_cols)
y_train = df_train_balanced["target"]

# === Rozszerzone parametry ===
param_distributions = {
    "n_estimators": [100, 200, 300, 500, 700],
    "learning_rate": [0.005, 0.01, 0.03, 0.05, 0.1, 0.2],
    "max_depth": [3, 4, 5, 6, 7, 9],
    "subsample": [0.5, 0.6, 0.8, 1.0],
    "min_samples_split": [2, 4, 5, 10, 15],
    "min_samples_leaf": [1, 2, 3, 5, 7],
    "max_features": ["sqrt", "log2", None]
}

# === Model ===
model = GradientBoostingClassifier(random_state=42)

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=50,
    scoring="f1_macro",
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

# === MLflow run ===
with mlflow.start_run(run_name="GBC_Undersampled_RandomSearch_2010-2024"):
    search.fit(X_train, y_train)

    best_params = search.best_params_
    best_score = search.best_score_
    best_model = search.best_estimator_

    # Log params and metrics
    mlflow.log_params(best_params)
    mlflow.log_metric("best_f1_macro", best_score)


    # Tagi pomocnicze
    mlflow.set_tag("strategy", "undersampling")
    mlflow.set_tag("model_type", "GradientBoostingClassifier")
    mlflow.set_tag("data_range", "2010–2024")

print("Gotowe")


#### GradientBoostingClassifier Training and Generating Prediction

In [ ]:
import pandas as pd
import json
from sklearn.ensemble import GradientBoostingClassifier
import mlflow
import mlflow.sklearn

# === MLflow setup ===
mlflow.set_tracking_uri("http://127.0.0.1:8080")
mlflow.set_experiment("NBA Final Prediction 2025")

with mlflow.start_run(run_name="GBC_Undersampled_2025"):
    # === 1. Wczytanie danych ===
    df_full = pd.read_csv("databases/nba_dataset_2010_2025.csv")

    # === 2. Undersampling danych treningowych ===
    df_majority = df_full[(df_full["season"] < 2025) & (df_full["target"] == 0)]
    df_minority = df_full[(df_full["season"] < 2025) & (df_full["target"] > 0)]

    df_majority_sampled = df_majority.sample(frac=0.5, random_state=42)
    df_train_balanced = pd.concat([df_majority_sampled, df_minority])

    # === 3. Przygotowanie cech i etykiet ===
    drop_cols = ["Player", "Team", "Pos", "season", "target"]
    X_train = df_train_balanced.drop(columns=drop_cols)
    y_train = df_train_balanced["target"]

    X_pred = df_full[df_full["season"] == 2025].drop(columns=drop_cols)
    players_pred = df_full[df_full["season"] == 2025]["Player"].reset_index(drop=True)
    is_rookie = df_full[df_full["season"] == 2025]["is_rookie"].reset_index(drop=True)

    # === 4. Trening modelu ===
    model = GradientBoostingClassifier(
        n_estimators=500,
        learning_rate=0.1,
        max_depth=9,
        subsample=0.5,
        random_state=42,
        min_samples_split=4,
        min_samples_leaf=3
    )

    mlflow.log_params(model.get_params())

    model.fit(X_train, y_train)

    # === 5. Predykcja ===
    probas = model.predict_proba(X_pred)

    # === 6. Zbudowanie piątek ===
    df_pred = pd.DataFrame(probas, columns=model.classes_)
    df_pred["Player"] = players_pred
    df_pred["is_rookie"] = is_rookie

    results = {
        "first all-nba team": df_pred.sort_values(1, ascending=False)["Player"].head(5).tolist(),
        "second all-nba team": df_pred.sort_values(2, ascending=False)["Player"].head(5).tolist(),
        "third all-nba team": df_pred.sort_values(3, ascending=False)["Player"].head(5).tolist(),
        "first rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(4, ascending=False)["Player"].head(5).tolist(),
        "second rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(5, ascending=False)["Player"].head(5).tolist()
    }

    # === 7. Zapis do JSON ===
    output_path = "classification_result_2025.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    mlflow.log_artifact(output_path)

    # === 8. Tagi i metryki pomocnicze ===
    mlflow.set_tag("model_type", "GradientBoostingClassifier")
    mlflow.set_tag("prediction_season", "2025")
    mlflow.set_tag("strategy", "undersampling")

print("Gotowe: zalogowano do MLflow i zapisano wynik.")


Uzyskany score:
- Total score: 256/450
- first all-nba team: 68 points
- second all-nba team: 48 points
- third all-nba team: 32 points
- first rookie all-nba team: 60 points
- second rookie all-nba team: 48 points

#### Cascade Model, GradientBoostingClassifier – Traning and Prediction 

In [ ]:
import pandas as pd
import json
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

# === 1. Dane ===
df = pd.read_csv("databases/nba_dataset_2010_2025.csv")
drop_cols = ["Player", "Team", "Pos", "season", "target"]

# === 2. Przygotowanie targetów binarnych (etap 1) ===
df["target_binary"] = (df["target"] > 0).astype(int)

# === 3. Podział danych ===
train_mask = df["season"] < 2025
test_mask = df["season"] == 2025

X_train = df[train_mask].drop(columns=drop_cols)
X_test = df[test_mask].drop(columns=drop_cols)

y_train_bin = df[train_mask]["target_binary"]
y_train_full = df[train_mask]["target"]
players_test = df[test_mask]["Player"].reset_index(drop=True)
is_rookie = df[test_mask]["is_rookie"].reset_index(drop=True)

# === 4. Model etapu 1 — nagroda vs brak ===
# === 4. Trening modelu ===
model_stage1 = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.2,
    max_depth=3,
    subsample=1.0,
    random_state=42,
    min_samples_split=10,
    min_samples_leaf=1
)
sample_weight = compute_sample_weight(class_weight="balanced", y=y_train_bin)
model_stage1.fit(X_train, y_train_bin, sample_weight=sample_weight)

# === 5. Predykcja — kto dostanie nagrodę? ===
X_pred = X_test.reset_index(drop=True)
stage1_preds = model_stage1.predict(X_pred)

# === 6. Etap 2 — tylko dla graczy z predykcją "nagroda" ===
X_train_stage2 = X_train[y_train_bin == 1]
y_train_stage2 = y_train_full[y_train_bin == 1]

X_test_stage2 = X_pred[stage1_preds == 1].reset_index(drop=True)
players_stage2 = players_test[stage1_preds == 1].reset_index(drop=True)
is_rookie_stage2 = is_rookie[stage1_preds == 1].reset_index(drop=True)

model_stage2 = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.2,
    max_depth=3,
    subsample=1.0,
    random_state=42,
    min_samples_split=10,
    min_samples_leaf=1
)
model_stage2.fit(X_train_stage2, y_train_stage2)

# === 7. Predykcja klas 1–5 tylko dla wybranych ===
probas_stage2 = model_stage2.predict_proba(X_test_stage2)
df_pred = pd.DataFrame(probas_stage2, columns=model_stage2.classes_)
df_pred["Player"] = players_stage2
df_pred["is_rookie"] = is_rookie_stage2

# === 8. Zbudowanie piątek ===
results = {
    "first all-nba team": df_pred.sort_values(1, ascending=False)["Player"].head(5).tolist(),
    "second all-nba team": df_pred.sort_values(2, ascending=False)["Player"].head(5).tolist(),
    "third all-nba team": df_pred.sort_values(3, ascending=False)["Player"].head(5).tolist(),
    "first rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(4, ascending=False)["Player"].head(5).tolist(),
    "second rookie all-nba team": df_pred[df_pred["is_rookie"] == 1].sort_values(5, ascending=False)["Player"].head(5).tolist()
}

# === 9. Zapis do JSON ===
with open("classification_result_2025.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)




Uzyskany score:

- Total score: 316/450
- first all-nba team: 68 points
- second all-nba team: 56 points
- third all-nba team: 56 points
- first rookie all-nba team: 68 points
- second rookie all-nba team: 68 points